# H4: Foraging Profiles and Optimality

- **H4a:** ω predicts escape on attack trials
- **H4b:** Overcaution is the dominant error; ω predicts who
- **H4c:** κ predicts pressing intensity
- **H4d:** Effort-driven avoidance is less optimal than threat-driven avoidance

Profiles (descriptive): Strategic, Resilient, Reckless, Helpless.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, os
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy.stats import pearsonr, zscore, f_oneway
from pathlib import Path

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10,
                     'axes.spines.right': False, 'axes.spines.top': False})

REPO_ROOT = Path(os.getcwd())
for _ in range(5):
    if (REPO_ROOT / '.git').exists(): break
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)

EXCLUDE = [154, 197, 208]
DATA_DIR = Path('data/exploratory_350/processed/stage5_filtered_data_20260320_191950')

# Load CM2 params
master = pd.read_csv('results/stats/joint_optimal/master_subject_df.csv', index_col='subj')
subjects = master.index.tolist()
master['omega_z'] = zscore(np.log(master['omega']))
master['kappa_z'] = zscore(np.log(master['kappa']))

# Load behavioral data for optimality analysis
beh = pd.read_csv(DATA_DIR / 'behavior_rich.csv', low_memory=False)
beh = beh[~beh['subj'].isin(EXCLUDE)]
cdf = beh[beh['type'] == 1].copy()
cdf['T_round'] = cdf['threat'].round(1)

# Escape rate (survived given attack)
attack = beh[beh['isAttackTrial'] == 1]
escape_rate = attack.groupby('subj').apply(lambda x: (x['trialEndState'] == 'escaped').mean())
master['escape_rate'] = escape_rate

# Earnings
master['earnings'] = beh.groupby('subj')['trialReward'].sum()

# Choice
master['p_heavy'] = cdf.groupby('subj')['choice'].mean()

# Vigor
cells = pd.read_csv('results/stats/vigor_analysis/cell_means.csv')
master['mean_vigor'] = cells.groupby('subj')['mean_rate'].mean()

# Optimality
opt_map = {}
for (T, D), grp in cdf.groupby(['T_round', 'distance_H']):
    er_h = grp[grp['choice']==1]['trialReward'].mean() if (grp['choice']==1).sum()>0 else -99
    er_l = grp[grp['choice']==0]['trialReward'].mean() if (grp['choice']==0).sum()>0 else -99
    opt_map[(T,D)] = 1 if er_h > er_l else 0
cdf['optimal'] = cdf.apply(lambda r: opt_map.get((r['T_round'],r['distance_H']),np.nan), axis=1)
cdf['is_opt'] = (cdf['choice'] == cdf['optimal']).astype(int)
cdf['err_type'] = np.where(cdf['is_opt']==1, 'correct',
                           np.where(cdf['choice']==0, 'overcautious', 'reckless'))
subj_opt = cdf.groupby('subj').agg(
    pct_opt=('is_opt','mean'),
    n_oc=('err_type', lambda x: (x=='overcautious').sum()),
    n_rk=('err_type', lambda x: (x=='reckless').sum()),
    n_err=('is_opt', lambda x: (x==0).sum()))
subj_opt['oc_ratio'] = subj_opt['n_oc'] / subj_opt['n_err'].clip(1)
# Drop any existing optimality columns before join
for c in subj_opt.columns:
    if c in master.columns:
        master = master.drop(columns=[c])
master = master.join(subj_opt, how='left')

print(f'Master: {len(master)} subjects')
print(f'  Escape: {master["escape_rate"].mean():.3f}, Earnings: {master["earnings"].mean():.1f}')

# Use MCMC posterior means if available (preferred over SVI)
mcmc_path = Path('results/stats/joint_optimal/mcmc_m5_params.csv')
if mcmc_path.exists():
    mcmc_p = pd.read_csv(mcmc_path).set_index('subj')
    shared = sorted(set(master.index) & set(mcmc_p.index))
    master.loc[shared, 'omega'] = mcmc_p.loc[shared, 'omega']
    master.loc[shared, 'kappa'] = mcmc_p.loc[shared, 'kappa']
    master['log_omega'] = np.log(master['omega'])
    master['log_kappa'] = np.log(master['kappa'])
    print(f'Updated {len(shared)} subjects with MCMC posterior means')


Master: 290 subjects
  Escape: 0.384, Earnings: 6.5
Updated 290 subjects with MCMC posterior means


## Descriptive: Four ω-κ profiles

In [2]:
om_med = master['omega_z'].median()
kap_med = master['kappa_z'].median()

conditions = [
    (master['omega_z'] < om_med) & (master['kappa_z'] < kap_med),
    (master['omega_z'] >= om_med) & (master['kappa_z'] < kap_med),
    (master['omega_z'] < om_med) & (master['kappa_z'] >= kap_med),
    (master['omega_z'] >= om_med) & (master['kappa_z'] >= kap_med),
]
labels = ['Resilient', 'Strategic', 'Reckless', 'Helpless']
master['profile'] = np.select(conditions, labels, default='?')

print('Descriptive: Four Foraging Profiles (not a preregistered test)')
print('=' * 70)
print(f'{"Profile":<12} {"N":>4} {"P(H)":>7} {"Escape":>8} {"Vigor":>8} {"Earn":>8}')
print('-' * 50)
for lab in ['Resilient', 'Strategic', 'Reckless', 'Helpless']:
    s = master[master['profile'] == lab]
    print(f'{lab:<12} {len(s):>4} {s["p_heavy"].mean():>7.3f} {s["escape_rate"].mean():>8.3f} '
          f'{s["mean_vigor"].mean():>8.3f} {s["earnings"].mean():>+8.1f}')

print(f'\nStrategic earns {master[master["profile"]=="Strategic"]["earnings"].mean() - master[master["profile"]=="Reckless"]["earnings"].mean():+.1f} more than Reckless.')

Descriptive: Four Foraging Profiles (not a preregistered test)
Profile         N    P(H)   Escape    Vigor     Earn
--------------------------------------------------
Resilient      96   0.619    0.334    1.024    +16.9
Strategic      49   0.366    0.439    1.224    +20.1
Reckless       49   0.500    0.346    0.782     -4.9
Helpless       96   0.235    0.424    0.879     -5.0

Strategic earns +25.0 more than Reckless.


## H4a: ω predicts escape on attack trials

In [3]:
X = sm.add_constant(master[['omega_z', 'kappa_z']].values)
y = master['escape_rate'].values
ols = sm.OLS(y, X).fit()

b_om = ols.params[1]; p_om = ols.pvalues[1]
b_kap = ols.params[2]; p_kap = ols.pvalues[2]
passed = b_om > 0 and p_om < 0.01

print('H4b — ω predicts escape')
print('=' * 50)
print(f'  ω: β={b_om:+.4f}, p={p_om:.4f}')
print(f'  κ: β={b_kap:+.4f}, p={p_kap:.4f}')
print(f'  R²={ols.rsquared:.4f}')
print(f'  H4b: {"PASS ✓" if passed else "FAIL ✗"} (threshold: ω β > 0, p < .01)')

H4b — ω predicts escape
  ω: β=+0.0586, p=0.0001
  κ: β=+0.0051, p=0.7323
  R²=0.0611
  H4b: PASS ✓ (threshold: ω β > 0, p < .01)


## H4b: Overcaution is the dominant error

In [4]:
oc_pct = master['oc_ratio'].mean() * 100
r_oc, p_oc = pearsonr(master['omega_z'], master['oc_ratio'])

pass_pct = oc_pct > 65
pass_r = r_oc > 0.30 and p_oc < 0.01
passed = pass_pct and pass_r

print('H4c — Overcaution')
print('=' * 50)
print(f'  Overcautious errors: {oc_pct:.0f}% of all errors {"✓" if pass_pct else "✗"} (threshold: >65%)')
print(f'  r(ω, overcaution ratio) = {r_oc:+.3f} (p={p_oc:.4f}) {"✓" if pass_r else "✗"} (threshold: r>.30, p<.01)')
print(f'  H4c: {"PASS ✓" if passed else "FAIL ✗"}')

H4c — Overcaution
  Overcautious errors: 79% of all errors ✓ (threshold: >65%)
  r(ω, overcaution ratio) = +0.784 (p=0.0000) ✓ (threshold: r>.30, p<.01)
  H4c: PASS ✓


## H4c: κ predicts pressing intensity

In [5]:
r_kv, p_kv = pearsonr(master['kappa_z'], master['mean_vigor'])
r_ov, p_ov = pearsonr(master['omega_z'], master['mean_vigor'])

passed = r_kv < -0.30 and p_kv < 0.01

print('H4c — κ predicts pressing intensity')
print('=' * 50)
print(f'  r(κ, mean vigor) = {r_kv:+.3f} (p={p_kv:.4f}) {"✓" if passed else "✗"}')
print(f'  r(ω, mean vigor) = {r_ov:+.3f} (p={p_ov:.4f}) (expected null)')
print(f'  H4c: {"PASS ✓" if passed else "FAIL ✗"} (threshold: r < -.30, p < .01)')

H4c — κ predicts pressing intensity
  r(κ, mean vigor) = -0.715 (p=0.0000) ✓
  r(ω, mean vigor) = +0.166 (p=0.0046) (expected null)
  H4c: PASS ✓ (threshold: r < -.30, p < .01)


## H4d: Effort-driven avoidance is less optimal than threat-driven

In [6]:
master['angle'] = np.arctan2(master['kappa_z'], master['omega_z'])

r_ao, p_ao = pearsonr(master['angle'], master['pct_opt'])
r_aoc, p_aoc = pearsonr(master['angle'], master['oc_ratio'])

passed = r_ao < -0.15 and p_ao < 0.01

print('H4d — ω-κ angle predicts optimality')
print('=' * 50)
print(f'  angle = atan2(κ_z, ω_z)  [high = effort-focused, low = threat-focused]')
print(f'  r(angle, optimality) = {r_ao:+.3f} (p={p_ao:.4f}) {"✓" if passed else "✗"}')
print(f'  r(angle, overcaution) = {r_aoc:+.3f} (p={p_aoc:.4f})')
print(f'  H4d: {"PASS ✓" if passed else "FAIL ✗"} (threshold: r < -.15, p < .01)')

H4d — ω-κ angle predicts optimality
  angle = atan2(κ_z, ω_z)  [high = effort-focused, low = threat-focused]
  r(angle, optimality) = -0.312 (p=0.0000) ✓
  r(angle, overcaution) = +0.445 (p=0.0000)
  H4d: PASS ✓ (threshold: r < -.15, p < .01)


## H4e: Consistency with W(u) predicts earnings

In [7]:
from scipy.special import expit

# Per-subject: model-predicted choice and u* for each condition
gamma=0.76; hazard=0.481; sp=0.25; C_pen=5.0
ug_np = np.linspace(0.1, 1.5, 100)

def compute_ustar_V(om, kap, T, D, R, req):
    speed = expit((ug_np - 0.25*req)/sp)
    S = np.exp(-hazard * T**gamma * D / np.clip(speed, 0.01, None))
    W = S*R - (1-S)*om*(R+C_pen) - kap*(ug_np-req)**2*D
    u_star = ug_np[W.argmax()]
    V = W.max() - kap * req * D  # total demand cost
    return u_star, V

# Choice consistency
choice_cons = {}
for s in master.index:
    om=master.loc[s,'omega']; kap=master.loc[s,'kappa']
    s_trials = cdf[cdf['subj']==s]; correct=0; total=0
    for _,trial in s_trials.iterrows():
        _,VH = compute_ustar_V(om,kap,trial['threat'],trial['distance_H'],5.,.9)
        _,VL = compute_ustar_V(om,kap,trial['threat'],1.,1.,.4)
        if (1 if VH>VL else 0)==int(trial['choice']): correct+=1
        total+=1
    choice_cons[s] = correct/total if total>0 else np.nan

# Intensity pattern match
int_pattern = {}
for s in master.index:
    om=master.loc[s,'omega']; kap=master.loc[s,'kappa']
    sc = cells[cells['subj']==s]
    if len(sc)<5: int_pattern[s]=np.nan; continue
    preds=[]; acts=[]
    for _,cell in sc.iterrows():
        R=5. if cell['is_heavy']==1 else 1.; req=0.9 if cell['is_heavy']==1 else 0.4
        u,_ = compute_ustar_V(om,kap,cell['T_round'],cell['actual_dist'],R,req)
        preds.append(u); acts.append(cell['mean_rate'])
    int_pattern[s] = pearsonr(preds, acts)[0]

master['choice_cons'] = pd.Series(choice_cons)
master['intensity_r'] = pd.Series(int_pattern)

v = master.dropna(subset=['choice_cons','intensity_r','earnings'])
from scipy.stats import zscore as zs
X = sm.add_constant(np.column_stack([zs(v['choice_cons']), zs(v['intensity_r'])]))
ols = sm.OLS(v['earnings'].values, X).fit()

r_cc = pearsonr(v['choice_cons'], v['intensity_r'])[0]
b_ch = ols.params[1]; p_ch = ols.pvalues[1]
b_vi = ols.params[2]; p_vi = ols.pvalues[2]
pass_ch = b_ch > 0 and p_ch < 0.01
pass_vi = b_vi > 0 and p_vi < 0.01
passed = pass_ch and pass_vi

print('H4e — Consistency with W(u) predicts earnings')
print('=' * 55)
print(f'  Choice consistency → earnings: r={pearsonr(v["choice_cons"],v["earnings"])[0]:+.3f}')
print(f'  Intensity pattern → earnings:  r={pearsonr(v["intensity_r"],v["earnings"])[0]:+.3f}')
print(f'  Joint R² = {ols.rsquared:.4f}')
print(f'    Choice:    β={b_ch:+.2f} (p={p_ch:.4f}) {"✓" if pass_ch else "✗"}')
print(f'    Intensity: β={b_vi:+.2f} (p={p_vi:.4f}) {"✓" if pass_vi else "✗"}')
print(f'  The two are independent: r={r_cc:+.3f}')
print(f'  H4e: {"PASS ✓" if passed else "FAIL ✗"} (both β > 0, both p < .01)')

H4e — Consistency with W(u) predicts earnings
  Choice consistency → earnings: r=+0.174
  Intensity pattern → earnings:  r=+0.426
  Joint R² = 0.2084
    Choice:    β=+14.26 (p=0.0021) ✓
    Intensity: β=+36.65 (p=0.0000) ✓
  The two are independent: r=+0.024
  H4e: PASS ✓ (both β > 0, both p < .01)


## Summary

In [8]:
tests = [
    ('H4a', 'ω → escape', 'β > 0, p < .01',
     f'β={b_om:+.4f}, p={p_om:.4f}', b_om > 0 and p_om < 0.01),
    ('H4b', 'Overcaution %', '> 65%',
     f'{oc_pct:.0f}%', oc_pct > 65),
    ('H4b', 'ω → overcaution', 'r > .30, p < .01',
     f'r={r_oc:+.3f}, p={p_oc:.4f}', r_oc > 0.30 and p_oc < 0.01),
    ('H4c', 'κ → vigor', 'r < -.30, p < .01',
     f'r={r_kv:+.3f}, p={p_kv:.4f}', r_kv < -0.30 and p_kv < 0.01),
    ('H4d', 'angle → optimality', 'r < -.15, p < .01',
     f'r={r_ao:+.3f}, p={p_ao:.4f}', r_ao < -0.15 and p_ao < 0.01),
    ('H4e', 'choice cons → earn', 'β > 0, p < .01',
     f'β={b_ch:+.2f}, p={p_ch:.4f}', b_ch > 0 and p_ch < 0.01),
    ('H4e', 'intensity → earn', 'β > 0, p < .01',
     f'β={b_vi:+.2f}, p={p_vi:.4f}', b_vi > 0 and p_vi < 0.01),
]

print('H4 SUMMARY')
print('=' * 70)
print(f'{"Hyp":<6} {"Test":<28} {"Threshold":<18} {"Result":<25} {"Pass":<5}')
print('-' * 80)
for hyp, test, thresh, result, passed in tests:
    print(f'{hyp:<6} {test:<28} {thresh:<18} {result:<25} {"✓" if passed else "✗"}')

n_pass = sum(1 for *_, p in tests if p)
print(f'\n{n_pass}/{len(tests)} tests pass.')

H4 SUMMARY
Hyp    Test                         Threshold          Result                    Pass 
--------------------------------------------------------------------------------
H4a    ω → escape                   β > 0, p < .01     β=+0.0586, p=0.0001       ✓
H4b    Overcaution %                > 65%              79%                       ✓
H4b    ω → overcaution              r > .30, p < .01   r=+0.784, p=0.0000        ✓
H4c    κ → vigor                    r < -.30, p < .01  r=-0.715, p=0.0000        ✓
H4d    angle → optimality           r < -.15, p < .01  r=-0.312, p=0.0000        ✓
H4e    choice cons → earn           β > 0, p < .01     β=+14.26, p=0.0021        ✓
H4e    intensity → earn             β > 0, p < .01     β=+36.65, p=0.0000        ✓

7/7 tests pass.
